<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Gradient Descent with Two Clock Settings

**Update the correction and hourly rate together using their slopes at the same candidate.**

Keep the four clock observations, corrected-error calculation, and squared-error objective. Remove the calibration equality and correction bound; the settings are unrestricted unless the rate is explicitly held at zero. Both settings vary in this notebook. The new calculation is a vector update using two partial derivatives.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Keep the decision and objective fixed

The observed clock errors are $d=(-2,-1,-1,0)$ min at times $t=(0,1,2,3)$ h. For observation $i=1,\ldots,4$, the corrected error is

$$
y_i=d_i+q+rt_i.
$$

The decision $x=(q,r)$ contains the initial correction $q$ in min and rate correction $r$ in min/h. The response vector is $y=\operatorname{Sim}(x)$. The scalar objective is the sum of squared errors:

$$
f(x)=\sum_{i=1}^{4}(d_i+q+rt_i)^2.
$$

We write $f(x)$ for the objective after composing the response calculation with the squared-error score. Its unit is $\mathrm{min}^2$. Every finite pair of real settings is feasible in this formulation. A smaller score therefore gives a better candidate.

This is a smooth, unconstrained least-squares problem and a quadratic program (QP). These names describe the problem form; gradient descent and Newton's method are solution procedures.

In the calculations, coordinates are numerical values in the stated units: $q=1$ means 1 min, and $r=1$ means 1 min/h. Gradient sizes and step sizes refer to this fixed coordinate convention. Changing units changes the coordinate scaling and can require a different step size.


In [ ]:
import numpy as np

# Fixed observations: each error corresponds to the time at the same position.
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def objective_function(response):
    return float(np.dot(response, response))


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("Use two finite numerical settings: correction and rate.")
    response = simulate_clock(decision)
    gradient = 2.0 * np.array([response.sum(), OBSERVATION_TIMES @ response])
    return {"decision": decision.copy(), "response": response,
            "feasible": True, "objective": objective_function(response),
            "gradient": gradient,
            "gradient_norm": float(np.linalg.norm(gradient))}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Update both settings together

Allow both the correction $q$ and the rate $r$ to vary. The observations, corrected-error equation, and sum of squares stay fixed. The decision is again $x=(q,r)$, with both components unrestricted.

The **gradient** collects the two partial derivatives, each calculated while holding the other setting fixed:

$$
\nabla f(q,r)=
\begin{bmatrix}
8q+12r-8\\
12q+28r-6
\end{bmatrix}.
$$

The superscript $k$ counts numerical updates, not elapsed clock time. The vector update uses both derivatives at the same current candidate:

$$
x^{(k+1)}=x^{(k)}-\alpha\nabla f(x^{(k)}).
$$

At $x^{(0)}=(0,0)$, the gradient is $(-8,-6)$. With $\alpha=0.05$, the first candidate is $(0.4,0.3)$, reducing the score from 6 to 4.34. At that new point the gradient is $(-1.2,7.2)$, so the second candidate is $(0.46,-0.06)$, with score 3.2456. The rate changes direction because its slope changes after the first update.

For a nonzero gradient, the directional slope along $-\nabla f$ is $-\|\nabla f\|_2^2<0$. This supports a sufficiently small descending step for a differentiable objective. It does not validate every fixed $\alpha$. In particular, the offset-only convergence range $0<α<0.25$, obtained with $r=0$, does not apply when both settings vary.

Both components update simultaneously, using derivatives at the same candidate. A numerical result includes its settings, objective value, and reason for stopping. The gradient tolerance $\varepsilon_g$ tests $\|\nabla f\|_2\le\varepsilon_g$ in the stated coordinates; the iteration limit $K$ caps computational work. Neither changes the clock's physical response.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The contours connect settings with the same squared-error score. The blue path shows 18 updates with $α=0.05$ from $(0,0)$; the teal marker is the stationary reference $(1.9,-0.6)$. The initial increase in rate is followed by alternating corrections.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/gradient_update_path.svg" alt="Contours of the two-setting clock objective show 18 gradient descent updates from zero. The first update increases both settings; later rate updates alternate before approaching the minimum at 1.9, minus 0.6." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>


In [ ]:
def run_gradient_descent(initial_decision=(0.0, 0.0), step_size=0.05,
                         gradient_tolerance=1e-6, max_steps=500):
    if not np.isfinite(step_size) or step_size <= 0:
        raise ValueError("The step size must be positive and finite.")
    if gradient_tolerance < 0 or max_steps < 0:
        raise ValueError("Use a nonnegative tolerance and iteration limit.")
    history = [evaluate_clock(initial_decision)]
    for iteration in range(max_steps + 1):
        current = history[-1]
        if current["gradient_norm"] <= gradient_tolerance:
            reason = "Gradient tolerance reached"
            break
        if iteration == max_steps:
            reason = "Iteration limit reached"
            break
        candidate = current["decision"] - step_size * current["gradient"]
        if not np.isfinite(candidate).all():
            reason = "Nonfinite candidate; update failed"
            break
        result = evaluate_clock(candidate)
        if not np.isfinite(result["objective"]) or not np.isfinite(result["gradient_norm"]):
            reason = "Nonfinite evaluation; update failed"
            break
        history.append(result)
    return {"result": history[-1], "history": history,
            "updates": len(history) - 1, "stop_reason": reason,
            "step_size": step_size, "gradient_tolerance": gradient_tolerance}

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Evaluate the update sequence

The first rows reproduce the two worked updates. The last row reports why the full run stops.


In [ ]:
gradient_run = run_gradient_descent()
for iteration, result in enumerate(gradient_run["history"][:3]):
    print(f"k={iteration}: x={result['decision']}, f={result['objective']:.4f} min²")
print("Final settings:", gradient_run["result"]["decision"])
print("Gradient norm:", gradient_run["result"]["gradient_norm"])
print("Updates:", gradient_run["updates"], "|", gradient_run["stop_reason"])

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Source material and scope

The supplied `Lecture_9_2_Traditional_Optimization02/9_2.pptx`, slides 4–8, introduces gradient-based methods, steepest descent, Newton's use of the Hessian, and initialization. Its `Example_9_2/hands_on_1.py` supplies a repeated negative-gradient update, a gradient-norm stopping test, and an iteration limit. This notebook applies simultaneous negative-gradient updates to both clock settings.

The source example's clipping operation is omitted because these settings are unrestricted. Plain gradient and Newton updates do not enforce a calibration equality or correction bound; restoring those requirements would require a constraint-aware method and explicit feasibility checks.
